# MobileNetV4-Small (Pruned + FT) → TFLite FP32
### Flutter / Android ready

**Steps:**
1. Run Cell 1 — installs dependencies  
2. Run Cell 2 — uploads your `finetuned_model.pth`  
3. Run Cells 3–7 — converts and verifies  
4. Run Cell 8 — downloads `finetuned_model.tflite`

> No GPU needed. CPU runtime is fine (`Runtime → Change runtime type → CPU`).

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Colab already ships torch and tensorflow.
# We pin specific versions so onnx-tf's op mapper works correctly with opset 13.
!pip install -q \
    timm==1.0.3 \
    onnx==1.16.2 \
    onnxruntime==1.18.0 \
    onnx-tf==1.10.0

print('\n✓ All dependencies ready')

In [ ]:
# ── Cell 2: Upload finetuned_model.pth ───────────────────────────────────────
from google.colab import files
import os

print('Click "Choose Files" and select your finetuned_model.pth:')
uploaded = files.upload()

pth_files = [f for f in uploaded.keys() if f.endswith('.pth')]
if not pth_files:
    raise ValueError('No .pth file detected. Please upload finetuned_model.pth')

MODEL_PATH = pth_files[0]
print(f'\n✓ Uploaded: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')

In [ ]:
# ── Cell 3: Load model and verify forward pass ────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import torch
import timm  # import before torch.load so timm classes can be unpickled

IMG_SIZE    = 224
NUM_CLASSES = 80

print('Loading model ...')
model = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
model.eval().cpu()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    pt_out = model(dummy).numpy()

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Parameters   : {params:.3f} M')
print(f'  Output shape : {pt_out.shape}')
print('✓ Model loaded')

In [ ]:
# ── Cell 4: PyTorch → ONNX ───────────────────────────────────────────────────
import onnx
import onnxruntime as ort

ONNX_PATH = '/content/model.onnx'

print('Exporting to ONNX (opset 13) ...')
torch.onnx.export(
    model, dummy, ONNX_PATH,
    opset_version       = 13,
    input_names         = ['input'],
    output_names        = ['logits'],
    dynamic_axes        = {'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    do_constant_folding = True,
)
print(f'  Saved ({os.path.getsize(ONNX_PATH)/1e6:.1f} MB)')

# Verify
onnx.checker.check_model(onnx.load(ONNX_PATH))
ort_out = ort.InferenceSession(
    ONNX_PATH, providers=['CPUExecutionProvider']
).run(None, {'input': dummy.numpy()})[0]
diff = float(np.abs(pt_out - ort_out).max())
print(f'  Max diff PyTorch vs ONNX : {diff:.2e}  {"✓" if diff < 1e-4 else "⚠ WARN"}')

In [ ]:
# ── Cell 5: ONNX → TF SavedModel ─────────────────────────────────────────────
from onnx_tf.backend import prepare as onnx_tf_prepare

TF_SAVED = '/content/tf_saved_model'

print('Converting ONNX → TF SavedModel (this may take ~60 s) ...')
tf_rep = onnx_tf_prepare(onnx.load(ONNX_PATH))
tf_rep.export_graph(TF_SAVED)
print(f'✓ TF SavedModel saved')

In [ ]:
# ── Cell 6: TF SavedModel → TFLite FP32 ──────────────────────────────────────
import tensorflow as tf

TFLITE_PATH = '/content/finetuned_model.tflite'

print(f'TensorFlow {tf.__version__}')
print('Converting → TFLite FP32 ...')

converter = tf.lite.TFLiteConverter.from_saved_model(TF_SAVED)
tflite_bytes = converter.convert()   # FP32 — no quantization flags

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_bytes)

print(f'✓ TFLite saved  ({os.path.getsize(TFLITE_PATH)/1e6:.1f} MB)')

In [ ]:
# ── Cell 7: Verify TFLite output ──────────────────────────────────────────────
print('Verifying TFLite output against PyTorch ...')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

inp_shape = tuple(inp_det['shape'])
print(f'  Input  : {inp_shape}  dtype={inp_det["dtype"].__name__}')
print(f'  Output : {tuple(out_det["shape"])}')

# ONNX → TF path typically preserves NCHW
if inp_shape == (1, 3, IMG_SIZE, IMG_SIZE):
    inp_arr = dummy.numpy()
    INPUT_LAYOUT = 'NCHW  [1, 3, 224, 224]'
else:
    inp_arr = dummy.permute(0, 2, 3, 1).numpy()
    INPUT_LAYOUT = 'NHWC  [1, 224, 224, 3]'

interp.set_tensor(inp_det['index'], inp_arr)
interp.invoke()
tflite_out = interp.get_tensor(out_det['index'])

max_diff = float(np.abs(pt_out - tflite_out).max())
status = '✓ PASS' if max_diff < 1e-3 else '⚠ large diff — double-check model'
print(f'\n  Max diff PyTorch vs TFLite : {max_diff:.2e}  {status}')
print(f'  Input layout for Flutter   : {INPUT_LAYOUT}')

In [ ]:
# ── Cell 8: Download ──────────────────────────────────────────────────────────
print('Starting download of finetuned_model.tflite ...')
files.download(TFLITE_PATH)

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Flutter / Android Integration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Input layout  : {INPUT_LAYOUT}
  Output        : Float32  [1, {NUM_CLASSES}]  class logits

  Pre-processing (ImageNet normalisation):
    r = (r/255 - 0.485) / 0.229
    g = (g/255 - 0.456) / 0.224
    b = (b/255 - 0.406) / 0.225

  pubspec.yaml:
    dependencies:
      tflite_flutter: ^0.10.4
    flutter:
      assets:
        - assets/finetuned_model.tflite

  Dart snippet:
    final interpreter = await Interpreter
        .fromAsset('finetuned_model.tflite');
    var output = List.filled({NUM_CLASSES}, 0.0).reshape([1, {NUM_CLASSES}]);
    interpreter.run(input, output);
    final classId = (output[0] as List<double>)
        .indexOf((output[0] as List<double>).reduce(max));
""")